# Pagtatayo ng Mga Aplikasyon sa Pagbuo ng Larawan (Azure OpenAI)

Hindi lamang teksto ang kayang gawin ng mga LLM. Maaari ka ring lumikha ng mga larawan mula sa mga paglalarawan ng teksto. Ang mga larawan bilang isang modality ay kapaki-pakinabang sa MedTech, arkitektura, turismo, pagbuo ng laro, marketing, at iba pa. Sa araling ito, gagamit tayo ng mga **GPT Image** na modelo sa Microsoft Foundry.

## Mga Layunin ng Pagkatuto

Sa pagtatapos ng araling ito ay magagawa mong:

- Ipaunawa kung ano ang pagbuo ng larawan at saan ito kapaki-pakinabang.
- Maunawaan ang pamilyang `gpt-image` na modelo at kung paano ito naiiba sa mga legacy na modelong DALL·E.
- Gumawa ng isang aplikasyon para sa pagbuo ng larawan at mag-edit ng mga larawan.

## Ano ang pagbuo ng larawan?

Ang mga modelo sa pagbuo ng larawan ay lumilikha ng mga larawan mula sa isang text prompt. Ang mga modernong modelo tulad ng `gpt-image` ay natututo ng ugnayan sa pagitan ng teksto at larawan habang nagsasanay, pagkatapos ay paulit-ulit na binabago ang random na ingay upang maging larawang tumutugma sa iyong prompt.

Dalawang kilalang pamilya ng mga modelong larawan ay:

- **`gpt-image` (OpenAI)** — ang kasalukuyang henerasyon na ginagamit sa araling ito. Sinusuportahan nito ang paggawa ng larawan mula sa teksto at pag-edit ng larawan (inpainting gamit ang mask).
- **Midjourney** — isang sikat na third-party na modelo na may sariling serbisyo at workflow sa Discord.

> Ang mga lumang modelo ng larawan ng OpenAI — **DALL·E 2** at **DALL·E 3** — ay legacy na. Ang DALL·E 3 ay hindi na available para sa mga bagong deployment, at ang mga tampok tulad ng `create_variation` ay nasa DALL·E 2 lamang. Gumamit ng mga `gpt-image` na modelo para sa mga bagong aplikasyon.

Sa Microsoft Foundry, ang **`gpt-image-2`** ang pinakabago at pinaka-kapangyarihang modelo ng larawan at ang inirerekomendang default. Ang `gpt-image-1.5` at `gpt-image-1-mini` ay karaniwan ding magagamit.

> **Mahalaga:** Ang mga `gpt-image` na modelo ay nagbabalik ng nabuo na larawan bilang **base64** (`b64_json`), hindi bilang URL. I-decode ng iyong code ang base64 string sa bytes at ise-save ito — walang image URL na pwedeng i-download.


## Paggawa ng iyong unang aplikasyon para sa pagbuo ng larawan

Ano nga ba ang kailangan upang makagawa ng aplikasyon para sa pagbuo ng larawan? Kailangan mo ang mga sumusunod na library:

- **python-dotenv**, lubos na inirerekomenda na gamitin ang library na ito upang itago ang iyong mga lihim sa isang *.env* na file na hiwalay sa code.
- **openai**, ito ang library na gagamitin mo upang makipag-ugnayan sa OpenAI API.
- **pillow**, para sa pagproseso ng mga larawan sa Python.

Kung hindi pa nagagawa, sundin ang mga tagubilin sa [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) na pahina para gumawa ng isang Microsoft Foundry resource at modelo. Piliin ang **gpt-image-2** bilang modelo (ang pinakabagong Azure OpenAI image model; ang DALL·E ay luma na).

1. Gumawa ng isang *.env* na file na may sumusunod na nilalaman:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Hanapin ang impormasyong ito sa Microsoft Foundry portal para sa iyong resource sa seksyong "Deployments."



1. Tipunin ang mga nabanggit na library sa isang file na tinawag na *requirements.txt* tulad nito:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Susunod, gumawa ng virtual environment at i-install ang mga library:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para sa Windows, gamitin ang mga sumusunod na command upang gumawa at i-activate ang iyong virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Idagdag ang sumusunod na code sa file na tinatawag na *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # i-configure ang Azure OpenAI (Microsoft Foundry) client.
    # Kailangan ng mga modelo ng imahe ang kamakailang bersyon ng API — tingnan ang mga dokumento ng Foundry para sa kinakailangan ng iyong modelo.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Gumawa ng imahe gamit ang image generation API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ilagay dito ang iyong prompt na teksto
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # hal. "gpt-image-2"
        )
        # Itakda ang direktoryo para sa naka-imbak na imahe
        image_dir = os.path.join(os.curdir, 'images')

        # Kung wala ang direktoryo, gawin ito
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # I-initialize ang path ng imahe (tandaan, ang uri ng file ay dapat png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Ang mga gpt-image na modelo ay nagbabalik ng imahe bilang base64 (b64_json), hindi URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Ipakita ang imahe sa default na viewer ng imahe
        image = Image.open(image_path)
        image.show()

    # hulihin ang mga exception
    except BadRequestError as err:
        print(err)

    ```

Ipaliwanag natin ang code na ito:

- Una, ini-import natin ang mga library na kailangan natin, kabilang ang OpenAI library, ang dotenv library, ang base64 na module, at ang Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Susunod, niloload natin ang mga environment variable mula sa *.env* na file.

    ```python
    # mag-import ng dotenv
    dotenv.load_dotenv()
    ```

- Pagkatapos noon, chine-configure natin ang Azure OpenAI (Microsoft Foundry) client.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Susunod, generate natin ang imahe:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ipasok ang iyong teksto ng prompt dito
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Ang mga `gpt-image` model ay nagbabalik ng imahe bilang isang **base64** string sa `data[0].b64_json`. Idecode natin ito sa bytes at isinusulat sa isang file — walang URL para i-download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Sa huli, binubuksan natin ang imahe at ginagamit ang standard image viewer upang ipakita ito:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Karagdagang detalye sa pag-generate ng imahe

Tingnan natin ang mga parameter ng `images.generate`:

- **prompt**, ay ang text prompt na ginagamit upang makabuo ng imahe. Dito, ito ay "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, ay ang sukat ng naregenerate na imahe (halimbawa `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, ay ang bilang ng mga imaheng naregenerate. Dito ay gumagawa tayo ng isa.
- **model**, ay ang pangalan ng iyong image model deployment (halimbawa `gpt-image-2`).

> Ang mga image model ay hindi tumatanggap ng `temperature` parameter — ito ay kontrol para sa text-generation. Para makakuha ng variety, tawagin ulit ang API; para mabawasan ang variety, gawing mas specific ang iyong prompt.

## Karagdagang kakayahan sa pag-generate ng imahe

Nakita mo na kung paano gumawa ng imahe gamit ang ilang linya ng Python. Ang mga `gpt-image` model ay kaya ring **mag-edit** ng umiiral na imahe. Sa pamamagitan ng pagbibigay ng isang imahe, isang optional **mask** (na nagtatalaga ng bahagi na babaguhin), at isang prompt, maaari mong baguhin ang bahagi ng isang imahe — halimbawa, magdagdag ng sombrero sa ating kuneho.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# ang mga pag-edit ay ibinabalik din bilang base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Ang base na imahe ay naglalaman lamang ng kuneho; ang huling imahe ay may sombrero na sa kuneho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
